# Speaker Diarization Pipeline Integration

Speaker diarization is wired into the Foreign Whispers pipeline so multi-speaker videos produce per-speaker labeled segments.

## Pipeline Flow

```
Download - Transcribe - Diarize - Translate - TTS (per-speaker voice cloning) - Stitch
```

## Prerequisites

- Docker Compose stack is running: `docker compose --profile nvidia up -d`
- `FW_HF_TOKEN` is set in your `.env` or environment
- You have accepted the [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) model license on HuggingFace
- A multi-speaker test video is in `video_registry.yml` and downloaded


## Provided Code (Read First)

| File | What it does |
| ---- | ------------ |
| `foreign_whispers/diarization.py` | `diarize_audio(audio_path, hf_token)` — calls pyannote, returns `[{start_s, end_s, speaker}]` |
| `api/src/services/alignment_service.py` | `AlignmentService.diarize()` — service wrapper |
| `api/src/core/config.py` | `Settings.hf_token` — reads `FW_HF_TOKEN` env var |
| `pipeline_data/speakers/` | Per-language directories with reference WAV files for TTS voice cloning |

In [ ]:
# Setup: add project root to path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

---

## Task 1: Segment-Speaker Merge Function

**Goal:** Write a pure function that assigns a speaker label to each transcription segment based on diarization output.

**Algorithm:** For each transcription segment, find which diarization speaker has the most temporal overlap with that segment's `[start, end]` range.

**File to modify:** `foreign_whispers/diarization.py` (add function to existing file)

### 1.1 — Read the existing diarization code

In [ ]:
# Read the existing diarization module
diar_path = PROJECT_ROOT / "foreign_whispers" / "diarization.py"
print(diar_path.read_text())

### 1.2 - Validate the merge behavior

Run the tests below as a quick notebook-level smoke test. They should pass against the implemented `assign_speakers` function.


In [ ]:
# These tests define the contract for assign_speakers.
# DO NOT modify the tests — make your implementation pass them.

def run_tests():
    from foreign_whispers.diarization import assign_speakers
    
    passed, failed = 0, 0
    
    # Test 1: Single speaker
    try:
        segments = [
            {"id": 0, "start": 0.0, "end": 3.0, "text": "Hello world"},
            {"id": 1, "start": 3.5, "end": 6.0, "text": "How are you"},
        ]
        diarization = [
            {"start_s": 0.0, "end_s": 7.0, "speaker": "SPEAKER_00"},
        ]
        result = assign_speakers(segments, diarization)
        assert len(result) == 2
        assert result[0]["speaker"] == "SPEAKER_00"
        assert result[1]["speaker"] == "SPEAKER_00"
        assert result[0]["text"] == "Hello world"
        print("✓ Test 1 passed: single speaker")
        passed += 1
    except Exception as e:
        print(f"✗ Test 1 FAILED: {e}")
        failed += 1
    
    # Test 2: Two speakers
    try:
        segments = [
            {"id": 0, "start": 0.0, "end": 4.0, "text": "Speaker A talking"},
            {"id": 1, "start": 5.0, "end": 9.0, "text": "Speaker B talking"},
            {"id": 2, "start": 10.0, "end": 14.0, "text": "Speaker A again"},
        ]
        diarization = [
            {"start_s": 0.0, "end_s": 4.5, "speaker": "SPEAKER_00"},
            {"start_s": 4.5, "end_s": 9.5, "speaker": "SPEAKER_01"},
            {"start_s": 9.5, "end_s": 15.0, "speaker": "SPEAKER_00"},
        ]
        result = assign_speakers(segments, diarization)
        assert result[0]["speaker"] == "SPEAKER_00"
        assert result[1]["speaker"] == "SPEAKER_01"
        assert result[2]["speaker"] == "SPEAKER_00"
        print("✓ Test 2 passed: two speakers")
        passed += 1
    except Exception as e:
        print(f"✗ Test 2 FAILED: {e}")
        failed += 1
    
    # Test 3: Empty diarization defaults to SPEAKER_00
    try:
        segments = [{"id": 0, "start": 0.0, "end": 3.0, "text": "Hello"}]
        result = assign_speakers(segments, [])
        assert result[0]["speaker"] == "SPEAKER_00"
        print("✓ Test 3 passed: empty diarization")
        passed += 1
    except Exception as e:
        print(f"✗ Test 3 FAILED: {e}")
        failed += 1
    
    # Test 4: Does not mutate input
    try:
        segments = [{"id": 0, "start": 0.0, "end": 3.0, "text": "Hello"}]
        original = segments[0].copy()
        assign_speakers(segments, [])
        assert segments[0] == original
        assert "speaker" not in segments[0]
        print("✓ Test 4 passed: no mutation")
        passed += 1
    except Exception as e:
        print(f"✗ Test 4 FAILED: {e}")
        failed += 1
    
    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run — expected to FAIL at this point
run_tests()

### 1.3 - `assign_speakers` Implementation

`foreign_whispers.diarization.assign_speakers()` is implemented. It copies each transcript segment and assigns the diarization speaker with the greatest temporal overlap, defaulting to `SPEAKER_00` when no overlap exists.


In [ ]:
import inspect
from foreign_whispers.diarization import assign_speakers

print(inspect.getsource(assign_speakers))


### 1.4 — Re-run the tests

After implementing, re-run the tests. All 4 should pass.

In [ ]:
# Reload the module and re-run
import importlib
import foreign_whispers.diarization
importlib.reload(foreign_whispers.diarization)

assert run_tests(), "Some tests failed — fix your implementation before continuing."

### 1.5 - Status

`assign_speakers` is implemented and covered by `tests/test_diarization.py`.


---

## Task 2: Diarize API Endpoint

**Goal:** Create `POST /api/diarize/{video_id}` that extracts audio, runs diarization, and returns speaker segments.

**Files to create:**
- `api/src/schemas/diarize.py`
- `api/src/routers/diarize.py`

**Files to modify:**
- `api/src/main.py` (register the router)
- `api/src/core/config.py` (add `diarizations_dir` property)

### 2.1 — Add `diarizations_dir` to Settings

Open `api/src/core/config.py` and add this property after `transcriptions_dir`:

```python
@property
def diarizations_dir(self) -> Path:
    return self.data_dir / "diarizations"
```

### 2.2 — Create the response schema

Create `api/src/schemas/diarize.py`:

In [ ]:
# Preview of the schema — create this file at api/src/schemas/diarize.py

schema_code = '''
"""Pydantic schemas for the diarize API contract."""

from pydantic import BaseModel


class DiarizeSpeakerSegment(BaseModel):
    start_s: float
    end_s: float
    speaker: str


class DiarizeResponse(BaseModel):
    video_id: str
    speakers: list[str]
    segments: list[DiarizeSpeakerSegment]
    skipped: bool = False
'''

schema_path = PROJECT_ROOT / "api" / "src" / "schemas" / "diarize.py"
schema_path.write_text(schema_code.strip() + "\n")
print(f"Created {schema_path}")

### 2.3 - Diarize Router (Implemented)

`api/src/routers/diarize.py` exposes `POST /api/diarize/{video_id}`. It extracts audio with ffmpeg, runs diarization through `AlignmentService`, caches results, merges speaker labels into transcript/translation JSON, and extracts one speaker reference WAV per diarized speaker.


In [ ]:
import inspect
import api.src.routers.diarize as diarize_router

print(inspect.getsource(diarize_router.diarize_endpoint))


### 2.4 — Register the router

Open `api/src/main.py` and add these lines alongside the existing router registrations (around line 94):

```python
from api.src.routers.diarize import router as diarize_router
app.include_router(diarize_router)
```

### 2.5 - Endpoint Behavior

The endpoint now handles both short and long videos:

1. Resolve `video_id` to title.
2. Reuse cached diarization if available.
3. Extract mono 16 kHz WAV audio via ffmpeg.
4. Chunk very long videos to avoid memory pressure.
5. Run pyannote diarization through `AlignmentService`.
6. Save diarization JSON.
7. Merge namespaced speaker labels into transcription and translation JSON.
8. Extract best per-speaker reference WAVs for TTS voice cloning.


### 2.6 — Rebuild and test

In [ ]:
# Rebuild the API container
!cd {PROJECT_ROOT} && docker compose --profile nvidia build api
!cd {PROJECT_ROOT} && docker compose --profile nvidia up -d api

In [ ]:
import requests

API_BASE = "http://localhost:8080"

# Get the first video ID from the registry
videos = requests.get(f"{API_BASE}/api/videos").json()
video_id = videos[0]["id"]
print(f"Testing with video: {videos[0]['title']} ({video_id})")

# Call the diarize endpoint
resp = requests.post(f"{API_BASE}/api/diarize/{video_id}")
print(f"Status: {resp.status_code}")
print(resp.json())

### 2.7 — Commit

```bash
git add api/src/schemas/diarize.py api/src/routers/diarize.py api/src/main.py api/src/core/config.py
git commit -m "feat: add POST /api/diarize endpoint with caching"
```

---

## Task 3: Merge Speaker Labels Into Transcription

**Goal:** After diarization runs, update the transcription JSON so each segment has a `speaker` field. This way downstream stages (translate, TTS) know which speaker said what.

**File to modify:** `api/src/routers/diarize.py`

### 3.1 — Add merge step to the diarize endpoint

After your diarization result is cached, add this code to update the transcription JSON with speaker labels:

```python
from foreign_whispers.diarization import assign_speakers

transcript_path = settings.transcriptions_dir / f"{title}.json"
if transcript_path.exists():
    transcript = json.loads(transcript_path.read_text())
    labeled_segments = assign_speakers(transcript.get("segments", []), diar_segments)
    transcript["segments"] = labeled_segments
    transcript_path.write_text(json.dumps(transcript))
```

### 3.2 — Verify the merge

In [ ]:
import json

# After running diarize, check the transcription for speaker labels
# Replace TITLE with your video's title
transcription_dir = PROJECT_ROOT / "pipeline_data" / "api" / "transcriptions" / "whisper"
transcription_files = list(transcription_dir.glob("*.json"))

if transcription_files:
    data = json.loads(transcription_files[0].read_text())
    print(f"File: {transcription_files[0].name}")
    print(f"Number of segments: {len(data.get('segments', []))}")
    print("\nFirst 3 segments:")
    for seg in data.get("segments", [])[:3]:
        speaker = seg.get("speaker", "<NO SPEAKER>")
        print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {speaker}: {seg['text'][:60]}")
else:
    print("No transcription files found — run the pipeline first.")

### 3.3 — Commit

```bash
git add api/src/routers/diarize.py
git commit -m "feat: merge speaker labels into transcription after diarization"
```

---

## Task 4: Frontend Pipeline Integration

**Goal:** Call the diarize endpoint from the frontend pipeline when diarization is enabled in settings.

**Files to modify:**
- `frontend/src/lib/api.ts` — add `diarizeVideo` function
- `frontend/src/lib/types.ts` — add `"diarize"` to `PipelineStage`
- `frontend/src/hooks/use-pipeline.ts` — call diarize between transcribe and translate
- `frontend/src/components/pipeline-table.tsx` — add diarize row
- `frontend/src/components/pipeline-status-bar.tsx` — add status message

### 4.1 — Add the API client function

In `frontend/src/lib/api.ts`, add:

```typescript
export async function diarizeVideo(videoId: string): Promise<DiarizeResponse> {
  return fetchJson<DiarizeResponse>(`/api/diarize/${videoId}`, {
    method: "POST",
  });
}
```

### 4.2 — Add TypeScript types

In `frontend/src/lib/types.ts`, add:

```typescript
export interface DiarizeResponse {
  video_id: string;
  speakers: string[];
  segments: { start_s: number; end_s: number; speaker: string }[];
  skipped: boolean;
}
```

Update `PipelineStage`:

```typescript
export type PipelineStage = "download" | "transcribe" | "diarize" | "translate" | "tts" | "stitch";
```

### 4.3 — Wire into the pipeline hook

In `frontend/src/hooks/use-pipeline.ts`, add the diarize call **between** transcribe and translate:

```typescript
// After: await run("transcribe", () => transcribeVideo(dl.video_id, settings.useYoutubeCaptions));
// Before: await run("translate", () => translateVideo(dl.video_id, "es"));

if (settings.diarization.length > 0) {
  await run("diarize", () => diarizeVideo(dl.video_id));
}
```

Don't forget to import `diarizeVideo` at the top of the file.

### 4.4 — Add UI elements

In `frontend/src/components/pipeline-table.tsx`, add a row to `STAGES`:

```typescript
import { UsersIcon } from "lucide-react";

// Add after "transcribe" entry:
{ key: "diarize", label: "Diarize", icon: UsersIcon, description: "Speaker diarization via pyannote" },
```

In `frontend/src/components/pipeline-status-bar.tsx`, add:

```typescript
diarize: "Running pyannote speaker diarization...",
```

### 4.5 — Build and test

In [ ]:
# Rebuild the frontend
!cd {PROJECT_ROOT} && docker compose --profile nvidia build frontend
!cd {PROJECT_ROOT} && docker compose --profile nvidia up -d frontend

print("\nOpen http://localhost:8501")
print("1. Click the gear icon to open Settings")
print("2. Go to TTS > Diarization and select 'pyannote'")
print("3. Run the pipeline — the Diarize stage should appear in the table")

### 4.6 — Commit

```bash
git add frontend/src/lib/api.ts frontend/src/lib/types.ts \
  frontend/src/hooks/use-pipeline.ts \
  frontend/src/components/pipeline-table.tsx \
  frontend/src/components/pipeline-status-bar.tsx
git commit -m "feat: add diarize stage to frontend pipeline"
```

---

## Task 5: Per-Speaker TTS Voice Selection (Implemented)

When speaker labels exist in translated segments, TTS now uses the diarized speaker reference WAV for that segment during Chatterbox voice cloning.

Files involved:

- `api/src/routers/diarize.py` extracts `pipeline_data/speakers/es/<video>__SPEAKER_XX.wav`
- `foreign_whispers/voice_resolution.py` resolves speaker WAVs
- `api/src/services/tts_engine.py` passes `speaker_wav` per segment and records `speaker_gender`
- `api/src/routers/tts.py` invalidates legacy cached audio that lacks speaker refs when voice cloning is requested


In [ ]:
# Explore available speaker reference voices
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
print("Available languages:")
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = list(lang_dir.glob("*.wav"))
        print(f"  {lang_dir.name}/  ({len(wavs)} reference WAVs)")
        for wav in wavs[:3]:
            print(f"    - {wav.name}")

### 5.1 - Voice Assignment Strategy

**Strategy chosen:** per-segment speaker reference resolution, with an explicit endpoint override for manual testing.

**Speaker ID flow:**

1. Pyannote emits local labels such as `SPEAKER_00` and `SPEAKER_01`.
2. The diarize router namespaces each label as `<video>__SPEAKER_XX` before merging it into transcription and translation JSON.
3. `_extract_speaker_references()` picks the longest usable diarized region for each speaker and writes `pipeline_data/speakers/es/<video>__SPEAKER_XX.wav`.
4. During TTS, each translated segment reads its own `segment["speaker"]` and resolves that to `speakers/{language}/{speaker}.wav`.

**Voice priority order:**

1. If the API caller passes `speaker_wav`, use that reference for every segment. This is useful for smoke tests and manual overrides.
2. Otherwise, if a segment has a diarized `speaker`, resolve that speaker-specific WAV and pass it to Chatterbox for that segment.
3. If the speaker-specific WAV is missing, fall back to the language default, then the global default when available.
4. If no usable WAV exists, synthesize with the default endpoint and keep the pipeline running.

**Gender handling:**

- Chatterbox gets the actual speaker reference WAV, so male and female speakers should inherit the corresponding source voice characteristics.
- `_speaker_gender()` estimates median F0 from the resolved speaker WAV and records `male`, `female`, or a deterministic fallback in the `.align.json` sidecar.
- The Flite fallback uses `FW_TTS_FLITE_MALE_VOICES` for male speakers and `FW_TTS_FLITE_FEMALE_VOICES` for female speakers, so fallback synthesis also preserves gender as much as possible.

**Cache and validation behavior:**

- TTS cache keys include `speaker_wav`, `speaker_voice`, `speaker_gender`, and `flite_voice`, so changing the resolved voice produces new raw phrase audio.
- The TTS router treats old diarized voice-cloning audio as stale when its sidecar has speaker labels but no `speaker_wav` entries.
- After running TTS, inspect `<output>.align.json`: each diarized segment should show `speaker`, `speaker_wav`, `speaker_gender`, and `speaker_voice`.
- For the original bug, the key check is that male speakers resolve to male reference WAVs and female speakers resolve to female reference WAVs instead of everyone using the same default voice.


**Design notes**

- Mapping shape: `translated segment["speaker"] -> pipeline_data/speakers/{language}/{speaker}.wav -> Chatterbox voice_file`.
- Namespacing avoids collisions when two videos both contain `SPEAKER_00`.
- The longest diarized region is preferred because short clips produce weaker voice references and less reliable gender estimates.
- Invalid or empty WAVs are handled inside `ChatterboxClient._synthesize_with_voice()`, which falls back to default speech rather than crashing.
- Long videos are chunked for diarization, then shifted back onto the original timeline before speaker refs are extracted.
- Stale cache protection matters because pre-fix audio may have `speaker_voice` metadata but no uploaded speaker reference.


### 5.2 - Implemented and Tested

The behavior is covered by focused TTS tests:

```bash
.venv\Scripts\python.exe -m pytest tests/test_diarization.py tests/test_tts_alignment_wire.py tests/test_tts_router.py
```

Use the voice mapping cell below after running diarization to inspect which speakers are present in the translated transcript and which reference WAVs are available.


---

## Evaluation Criteria

| # | Criterion | How to verify |
| - | --------- | ------------- |
| 1 | Tests pass | Re-run the test cell in Task 1.4 — all 4 green |
| 2 | API works | `POST /api/diarize/{video_id}` returns speaker segments |
| 3 | Merge works | Transcription JSON has `speaker` fields after diarization |
| 4 | Frontend works | Diarize stage appears in pipeline table when enabled |
| 5 | Caching works | Second API call returns `skipped: true` |
| 6 | Code quality | Follows existing patterns (file-exists cache, service layer, Pydantic schemas) |